[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/03_RQ1_Architecture_Comparison.ipynb)

# Notebook 03 — RQ1: Architecture Comparison

> **RQ1.** Which neural architecture (feed-forward network, one-dimensional CNN, or bidirectional LSTM) provides the best trade-off between predictive accuracy and computational cost for review sentiment classification, relative to a TF-IDF logistic regression baseline?

| Model | Input | Reference |
|---|---|---|
| TF-IDF + logistic regression (baseline) | TF-IDF unigrams + bigrams (50,000, min df 3, sublinear tf) | Wang & Manning (2012) |
| Feed-forward network 256-64-1 | Same TF-IDF features | Rumelhart et al. (1986) |
| 1D-CNN, filters 3/4/5 × 100 | Token ids (vocab 20,000, length 400), embeddings 128 | Kim (2014) |
| BiLSTM, 64 units per direction | Same token ids | Hochreiter & Schmidhuber (1997) |

**What is new compared with the interim report**
* Neural models are run with **5 random seeds** on a GPU, and the results are reported as mean ± SD.
* There is a small **hyperparameter search** on the validation set (learning rate, dropout, depth/width, weight decay). Early stopping now monitors **validation loss**, as planned to counter the overfitting seen in Figure 7.
* The CNN and BiLSTM get a longer training budget.
* The final evaluation is on the **untouched official IMDb test set** (~25,000 reviews).
* The **score-masking sensitivity** test runs for all four models. The score-masked models are saved for the transfer experiments in RQ2.

⚙️ **Runtime → Change runtime type → T4 GPU** before running. Every run is saved as soon as it finishes, so after a disconnect the notebook can simply be re-run and it skips completed runs.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

### 0.1 Shared PyTorch module (`code/cvi_models.py`)
Model classes, the training loop and prediction helpers. Notebooks 04–06 import this file.

In [ ]:
%%writefile {PROJECT_DIR}/code/cvi_models.py
"""PyTorch models and training utilities for RQ1 (also used in RQ2-RQ4)."""
import copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import log_loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FFN(nn.Module):
    """Feed-forward network on TF-IDF features (Rumelhart et al., 1986)."""
    def __init__(self, n_in, h1=256, h2=64, dropout=0.5):
        super().__init__()
        layers = [nn.Linear(n_in, h1), nn.ReLU(), nn.Dropout(dropout)]
        if h2:
            layers += [nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(h2, 1)]
        else:
            layers += [nn.Linear(h1, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x, lengths=None):
        return self.net(x).squeeze(-1)


class KimCNN(nn.Module):
    """1D-CNN over word embeddings with several filter widths (Kim, 2014)."""
    def __init__(self, vocab_size, emb=128, n_filters=100, widths=(3, 4, 5), dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, n_filters, w) for w in widths])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(widths), 1)

    def forward(self, x, lengths=None):
        e = self.emb(x).transpose(1, 2)                       # (B, E, T)
        pooled = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        return self.fc(self.drop(torch.cat(pooled, dim=1))).squeeze(-1)


class BiLSTM(nn.Module):
    """Bidirectional LSTM; final states + masked max-pool over time (Hochreiter & Schmidhuber, 1997)."""
    def __init__(self, vocab_size, emb=128, hidden=64, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(4 * hidden, 1)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, _) = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths.to(x.device)[:, None]
        maxpool = out.masked_fill(~mask[:, :, None], -1e9).max(dim=1).values
        final = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(torch.cat([final, maxpool], dim=1))).squeeze(-1)


def make_model(kind, cfg, n_in=None, vocab_size=None):
    if kind == "ffn":
        return FFN(n_in, cfg["h1"], cfg["h2"], cfg["dropout"])
    if kind == "cnn":
        return KimCNN(vocab_size, cfg["emb"], cfg["n_filters"], tuple(cfg["widths"]), cfg["dropout"])
    if kind == "bilstm":
        return BiLSTM(vocab_size, cfg["emb"], cfg["hidden"], cfg["dropout"])
    raise ValueError(kind)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# data = scipy sparse / numpy matrix (FFN)   or   tuple (ids, lengths) (CNN, BiLSTM)
def n_rows(data):
    return data[0].shape[0] if isinstance(data, tuple) else data.shape[0]


def get_batch(data, idx, device=None):
    device = device or DEVICE
    if isinstance(data, tuple):
        ids, lens = data
        return torch.from_numpy(ids[idx]).to(device), torch.from_numpy(lens[idx]).to(device)
    xb = data[idx]
    xb = xb.toarray() if hasattr(xb, "toarray") else xb
    return (torch.from_numpy(np.asarray(xb, dtype=np.float32)).to(device), None)


@torch.no_grad()
def predict_proba(model, data, batch_size=512, device=None):
    device = device or DEVICE
    model.eval()
    out = []
    n = n_rows(data)
    for i in range(0, n, batch_size):
        idx = np.arange(i, min(i + batch_size, n))
        x, lens = get_batch(data, idx, device)
        out.append(torch.sigmoid(model(x, lens)).float().cpu().numpy())
    return np.concatenate(out)


def train_model(model, train_data, y_train, val_data, y_val, lr=1e-3, batch_size=64, max_epochs=10,
                patience=2, weight_decay=0.0, seed=42, clip=5.0, verbose=True):
    """Adam + binary cross-entropy; early stopping on validation loss; best weights restored."""
    torch.manual_seed(seed); np.random.seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    rng = np.random.RandomState(seed)
    y_train = np.asarray(y_train, dtype=np.float32); y_val = np.asarray(y_val)
    n = n_rows(train_data)
    best, best_state, bad, hist = np.inf, None, 0, []
    t0 = time.perf_counter()
    for ep in range(1, max_epochs + 1):
        model.train(); perm = rng.permutation(n); total = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            x, lens = get_batch(train_data, idx)
            yb = torch.from_numpy(y_train[idx]).to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x, lens), yb)
            loss.backward()
            if clip:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            total += loss.item() * len(idx)
        p_val = predict_proba(model, val_data)
        v_loss = log_loss(y_val, np.clip(p_val, 1e-7, 1 - 1e-7), labels=[0, 1])
        v_acc = float(((p_val >= 0.5).astype(int) == y_val).mean())
        hist.append(dict(epoch=ep, train_loss=total / n, val_loss=v_loss, val_acc=v_acc,
                         elapsed_min=(time.perf_counter() - t0) / 60))
        if verbose:
            print(f"    epoch {ep:2d}  train_loss {total / n:.4f}  val_loss {v_loss:.4f}  val_acc {v_acc:.4f}")
        if v_loss < best - 1e-4:
            best, bad = v_loss, 0
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, hist

In [ ]:
import importlib, json, time, copy, glob
import cvi_utils as U, cvi_models as M
importlib.reload(U); importlib.reload(M)
U.set_project(PROJECT_DIR)
import numpy as np, pandas as pd, joblib, torch
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
U.plot_style()
print("Device:", M.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if not torch.cuda.is_available():
    print("WARNING: no GPU found. The notebook still runs, but the CNN and BiLSTM will be slow.")

## 1. Configuration

In [ ]:
SEEDS = [42, 43, 44, 45, 46]
MAX_VOCAB, MAX_LEN = 20000, 400                       # from Figure 3
TFIDF_KW = dict(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True)
C_GRID = [0.5, 1, 2, 4, 8, 16, 32]                     # extended beyond C = 8 (interim next step)

# Base configurations (Appendix B) + early stopping on validation loss
BASE = {
    "ffn":    dict(h1=256, h2=64, dropout=0.5, lr=1e-3, batch_size=128, max_epochs=8, weight_decay=1e-5),
    "cnn":    dict(emb=128, n_filters=100, widths=[3, 4, 5], dropout=0.5, lr=1e-3, batch_size=64, max_epochs=15, weight_decay=0.0),
    "bilstm": dict(emb=128, hidden=64, dropout=0.5, lr=2e-3, batch_size=64, max_epochs=12, weight_decay=0.0),
}
# Small validation-set grid (seed 42); each entry overrides the base configuration
GRID = {
    "ffn":    [{}, dict(h1=128, h2=0), dict(weight_decay=1e-4), dict(dropout=0.7, weight_decay=1e-4)],
    "cnn":    [{}, dict(lr=5e-4), dict(dropout=0.3)],
    "bilstm": [{}, dict(lr=1e-3), dict(hidden=128)],
}
PATIENCE = 2
NAMES = {"logreg": "TF-IDF + LogReg", "ffn": "Feed-forward NN", "cnn": "1D-CNN", "bilstm": "BiLSTM"}

# Cost assumptions used for cost per 1,000 reviews (update with current cloud prices; also used in Notebook 06)
COSTS = dict(cpu_usd_per_hour=0.05, gpu_usd_per_hour=0.35, cpu_threads=2,
             note="Assumed on-demand prices for a 2-vCPU instance and a T4 GPU instance; update before the final report.")

if U.QUICK:
    SEEDS = [42, 43]; C_GRID = [1, 8]
    for k in BASE: BASE[k]["max_epochs"] = 2
    GRID = {k: [{}] for k in GRID}

RES = U.P("outputs", "results"); os.makedirs(f"{RES}/rq1_probs", exist_ok=True)
MOD = U.P("models", "rq1"); os.makedirs(MOD, exist_ok=True)
json.dump(COSTS, open(f"{RES}/cost_assumptions.json", "w"), indent=2)
RUNS_CSV = f"{RES}/rq1_runs.csv"

## 2. Data and features
Two text variants are prepared:
* **raw**: the cleaned text, used for the main benchmark;
* **masked**: explicit scores such as "7/10" replaced by a placeholder. This variant is used for the sensitivity test and for the models that are transferred to Amazon in RQ2.

The TF-IDF vectorizer and the 20,000-word vocabulary are fitted on the **training split only**.

In [ ]:
imdb = U.load_imdb()
SPLITS = ["train", "validation", "dev_test", "test"]
y = {s: imdb.loc[imdb.split == s, "label"].to_numpy() for s in SPLITS}
meta = {s: imdb.loc[imdb.split == s, ["review_id", "label", "token_count", "has_score"]].reset_index(drop=True) for s in SPLITS}
LONG_CUT = int(imdb.loc[imdb.split == "train", "token_count"].median())
print({s: len(v) for s, v in y.items()}, "| long-review cut-off (training median):", LONG_CUT, "tokens")

DATA = {}
for variant in ["raw", "masked"]:
    texts = {s: imdb.loc[imdb.split == s, "review_text"].tolist() for s in SPLITS}
    if variant == "masked":
        texts = {s: [U.mask_scores(t) for t in v] for s, v in texts.items()}
    with U.Timer() as t:
        vec = TfidfVectorizer(tokenizer=U.tokenize, token_pattern=None, lowercase=False, dtype=np.float32, **TFIDF_KW)
        Xtr = vec.fit_transform(texts["train"])
    tfidf_fit_sec = t.seconds
    X = {"train": Xtr, **{s: vec.transform(texts[s]) for s in SPLITS[1:]}}
    vocab = U.build_vocab(texts["train"], MAX_VOCAB)
    seq = {s: U.encode(texts[s], vocab, MAX_LEN) for s in SPLITS}
    DATA[variant] = dict(texts=texts, vec=vec, X=X, vocab=vocab, seq=seq, tfidf_fit_sec=tfidf_fit_sec)
    joblib.dump(vec, f"{MOD}/tfidf_{variant}.joblib")
    json.dump(vocab, open(f"{MOD}/vocab_{variant}.json", "w"))
    print(f"{variant}: TF-IDF {Xtr.shape[1]:,} features (fit {tfidf_fit_sec:.1f}s); vocab {len(vocab):,}; "
          f"truncated at {MAX_LEN}: {(seq['train'][1] >= MAX_LEN).mean():.1%}")

## 3. Experiment runner
Each run stores its validation, development-test and official-test probabilities. Before running, the notebook checks whether a run already exists.

In [ ]:
def done_runs():
    return pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=["key"])

def evaluate(row, probs):
    for s in ["dev_test", "test"]:
        m = U.classification_metrics(y[s], probs[s])
        for k in ["accuracy", "macro_f1", "roc_auc", "precision", "recall"]:
            row[f"{s}_{k}"] = m[k]
        long_ = meta[s].token_count.to_numpy() > LONG_CUT
        pred = (probs[s] >= 0.5).astype(int)
        row[f"{s}_acc_long"] = (pred[long_] == y[s][long_]).mean()
        row[f"{s}_acc_short"] = (pred[~long_] == y[s][~long_]).mean()
        sc = meta[s].has_score.to_numpy() == 1
        row[f"{s}_acc_score_reviews"] = (pred[sc] == y[s][sc]).mean() if sc.any() else np.nan
        row[f"{s}_acc_noscore_reviews"] = (pred[~sc] == y[s][~sc]).mean()
    return row

def save_run(row, probs, hist=None):
    np.savez_compressed(f"{RES}/rq1_probs/{row['key']}.npz", **probs)
    if hist is not None:
        pd.DataFrame(hist).assign(key=row["key"]).to_csv(f"{RES}/rq1_probs/{row['key']}_history.csv", index=False)
    U.append_csv(row, RUNS_CSV)

def run_logreg(variant):
    key = f"logreg_{variant}_s42"
    if key in set(done_runs().key):
        print("skip", key); return
    D = DATA[variant]
    with U.Timer() as t:
        best_c, best_acc = None, -1
        for c in C_GRID:
            m = LogisticRegression(C=c, solver="liblinear", max_iter=2000).fit(D["X"]["train"], y["train"])
            acc = m.score(D["X"]["validation"], y["validation"])
            print(f"  C = {c:<5} validation accuracy {acc:.4f}")
            if acc > best_acc:
                best_c, best_acc, best = c, acc, m
    probs = {s: best.predict_proba(D["X"][s])[:, 1] for s in ["validation", "dev_test", "test"]}
    row = dict(key=key, model="logreg", variant=variant, seed=42, config=json.dumps({"C": best_c}),
               train_time_min=(t.seconds / len(C_GRID) + D["tfidf_fit_sec"]) / 60,   # one fit + feature extraction
               tuning_time_min=t.seconds / 60, epochs=np.nan, n_params=best.coef_.size + 1, val_accuracy=best_acc)
    save_run(evaluate(row, probs), probs)
    joblib.dump(best, f"{MOD}/logreg_{variant}_s42.joblib")
    print(f"{key}: best C = {best_c}, test accuracy {row['test_accuracy']:.4f}")

def run_neural(kind, cfg, seed, variant, tag="final", save_model=False):
    key = f"{kind}_{variant}_s{seed}" if tag == "final" else f"tune_{kind}_{tag}"
    if key in set(done_runs().key):
        print("skip", key); return None
    D = DATA[variant]
    data = {s: D["X"][s] for s in SPLITS} if kind == "ffn" else D["seq"]
    U.set_seed(seed)
    model = M.make_model(kind, cfg, n_in=D["X"]["train"].shape[1], vocab_size=len(D["vocab"]))
    print(f"{key}  ({M.count_params(model) / 1e6:.2f}M parameters)  {cfg}")
    with U.Timer() as t:
        model, hist = M.train_model(model, data["train"], y["train"], data["validation"], y["validation"],
                                    lr=cfg["lr"], batch_size=cfg["batch_size"], max_epochs=cfg["max_epochs"],
                                    patience=PATIENCE, weight_decay=cfg["weight_decay"], seed=seed)
    extra = D["tfidf_fit_sec"] if kind == "ffn" else 0.0
    probs = {s: M.predict_proba(model, data[s]) for s in ["validation", "dev_test", "test"]}
    row = dict(key=key, model=kind, variant=variant, seed=seed, config=json.dumps(cfg),
               train_time_min=(t.seconds + extra) / 60, tuning_time_min=np.nan, epochs=len(hist),
               n_params=M.count_params(model), val_accuracy=U.classification_metrics(y["validation"], probs["validation"])["accuracy"])
    save_run(evaluate(row, probs), probs, hist)
    if save_model:
        torch.save(model.state_dict(), f"{MOD}/{kind}_{variant}_s{seed}.pt")
        json.dump(dict(kind=kind, cfg=cfg, n_in=D["X"]["train"].shape[1], vocab_size=len(D["vocab"]), variant=variant),
                  open(f"{MOD}/{kind}_{variant}_s{seed}.json", "w"))
    print(f"   -> val acc {row['val_accuracy']:.4f} | test acc {row['test_accuracy']:.4f} | {row['train_time_min']:.1f} min")
    del model; torch.cuda.empty_cache()
    return row

## 4. Baseline: TF-IDF + logistic regression
The regularization strength C is tuned on the validation set.

In [ ]:
for variant in ["raw", "masked"]:
    print("Variant:", variant); run_logreg(variant)

## 5. Hyperparameter search for the neural models (validation set, seed 42)

In [ ]:
for kind in ["ffn", "cnn", "bilstm"]:
    for i, override in enumerate(GRID[kind]):
        run_neural(kind, {**BASE[kind], **override}, 42, "raw", tag=f"g{i}")

tune = done_runs()
tune = tune[tune.key.str.startswith("tune_")].copy()
BEST_CFG = {}
for kind in ["ffn", "cnn", "bilstm"]:
    t = tune[tune.model == kind].sort_values("val_accuracy", ascending=False)
    BEST_CFG[kind] = json.loads(t.iloc[0].config)
    print(f"{NAMES[kind]:16s} best validation accuracy {t.iloc[0].val_accuracy:.4f} with {BEST_CFG[kind]}")
json.dump(BEST_CFG, open(f"{RES}/rq1_best_configs.json", "w"), indent=2)
tuning_table = tune[["model", "config", "val_accuracy", "epochs", "train_time_min"]].sort_values(["model", "val_accuracy"], ascending=[True, False])
U.save_table(tuning_table, "rq1_tuning_results")
tuning_table

## 6. Final runs: five seeds per neural model
The seed-42 model of each architecture is saved for use in RQ2–RQ4.

In [ ]:
for kind in ["ffn", "cnn", "bilstm"]:
    for seed in SEEDS:
        run_neural(kind, BEST_CFG[kind], seed, "raw", save_model=(seed == 42))

## 7. Score-masking sensitivity (seed 42, all models)

In [ ]:
for kind in ["ffn", "cnn", "bilstm"]:
    run_neural(kind, BEST_CFG[kind], 42, "masked", save_model=True)

## 8. Inference latency and cost
Latency is measured **end to end**, from raw text to probability, including tokenisation and TF-IDF transformation. The measurement uses a fixed sample of 1,000 test reviews on the CPU, limited to two threads to match a small 2-vCPU instance. GPU throughput is also measured for the neural models. Cost per 1,000 reviews = hourly price × hours needed to score 1,000 reviews.

In [ ]:
LAT_N = 1000 if not U.QUICK else 200
lat_texts = DATA["raw"]["texts"]["test"][:LAT_N]
D = DATA["raw"]

def load_torch(kind, variant="raw", seed=42, device="cpu"):
    spec = json.load(open(f"{MOD}/{kind}_{variant}_s{seed}.json"))
    m = M.make_model(kind, spec["cfg"], spec["n_in"], spec["vocab_size"])
    m.load_state_dict(torch.load(f"{MOD}/{kind}_{variant}_s{seed}.pt", map_location=device))
    return m.to(device).eval()

def e2e(kind, model, texts, device, batch=256):
    if kind == "logreg":
        return model.predict_proba(D["vec"].transform(texts))[:, 1]
    data = D["vec"].transform(texts) if kind == "ffn" else U.encode(texts, D["vocab"], MAX_LEN)
    return M.predict_proba(model, data, batch_size=batch, device=torch.device(device))

def time_it(fn, repeats=3):
    fn()                                  # warm-up
    ts = []
    for _ in range(repeats):
        with U.Timer() as t: fn()
        ts.append(t.seconds)
    return float(np.median(ts))

n_threads = torch.get_num_threads(); torch.set_num_threads(COSTS["cpu_threads"])
lat_rows = []
for kind in ["logreg", "ffn", "cnn", "bilstm"]:
    model = joblib.load(f"{MOD}/logreg_raw_s42.joblib") if kind == "logreg" else load_torch(kind, device="cpu")
    sec = time_it(lambda: e2e(kind, model, lat_texts, "cpu"))
    row = dict(model=kind, cpu_ms_per_review=1000 * sec / LAT_N)
    if kind != "logreg" and torch.cuda.is_available():
        gm = load_torch(kind, device="cuda")
        gsec = time_it(lambda: (e2e(kind, gm, lat_texts, "cuda"), torch.cuda.synchronize()))
        row["gpu_ms_per_review"] = 1000 * gsec / LAT_N
    lat_rows.append(row)
torch.set_num_threads(n_threads)
lat = pd.DataFrame(lat_rows)
lat["cost_1k_cpu_usd"] = lat.cpu_ms_per_review / 1000 * 1000 / 3600 * COSTS["cpu_usd_per_hour"]
if "gpu_ms_per_review" in lat:
    lat["cost_1k_gpu_usd"] = lat.gpu_ms_per_review / 1000 * 1000 / 3600 * COSTS["gpu_usd_per_hour"]
lat.to_csv(f"{RES}/rq1_latency.csv", index=False)
lat

## 9. Results
### Table 7 — Model performance on the official IMDb test set

In [ ]:
runs = done_runs()
final = runs[~runs.key.str.startswith("tune_") & (runs.variant == "raw")].copy()
n_test = len(y["test"])

def mean_sd(s):
    return f"{100 * s.mean():.2f}%" + (f" ± {100 * s.std(ddof=1):.2f}" if len(s) > 1 else "")

rows = []
for kind in ["logreg", "ffn", "cnn", "bilstm"]:
    r = final[final.model == kind]
    l = lat.set_index("model").loc[kind]
    lo, hi = U.accuracy_ci(r.test_accuracy.mean(), n_test)
    rows.append({
        "Model": NAMES[kind], "Accuracy": mean_sd(r.test_accuracy), "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]",
        "Macro F1": f"{r.test_macro_f1.mean():.4f}", "ROC-AUC": f"{r.test_roc_auc.mean():.4f}",
        "Acc. long": f"{100 * r.test_acc_long.mean():.1f}%", "Acc. short": f"{100 * r.test_acc_short.mean():.1f}%",
        "Train time (min)": f"{r.train_time_min.mean():.1f}", "Epochs": "" if kind == "logreg" else f"{r.epochs.mean():.1f}",
        "Latency CPU (ms/review)": f"{l.cpu_ms_per_review:.2f}",
        "Latency GPU (ms/review)": "" if pd.isna(l.get("gpu_ms_per_review", np.nan)) else f"{l.gpu_ms_per_review:.3f}",
        "Cost / 1k reviews (USD, CPU)": f"{l.cost_1k_cpu_usd:.5f}",
        "Params (M)": f"{r.n_params.iloc[0] / 1e6:.2f}", "Seeds": len(r),
    })
table7 = pd.DataFrame(rows)
U.save_table(table7, "table7_rq1_performance_test")
print(f"Official test set n = {n_test:,}; long reviews = above {LONG_CUT} tokens (training median)")
table7

In [ ]:
# Same table on the development test set (comparable with the interim report)
dev = (final.groupby("model")[["dev_test_accuracy", "dev_test_macro_f1", "dev_test_roc_auc", "dev_test_acc_long"]]
       .agg(["mean", "std"]).round(4).reindex(["logreg", "ffn", "cnn", "bilstm"]))
dev.index = [NAMES[k] for k in dev.index]
U.save_table(dev, "table7b_rq1_performance_devtest", index=True)
dev

### Table 8 — Paired McNemar tests (seed 42, official test set)

In [ ]:
def load_probs(key):
    return np.load(f"{RES}/rq1_probs/{key}.npz")

pred42 = {k: (load_probs(f"{k}_raw_s42")["test"] >= 0.5).astype(int) for k in ["logreg", "ffn", "cnn", "bilstm"]}
pairs = [("logreg", "ffn"), ("logreg", "cnn"), ("logreg", "bilstm"), ("ffn", "cnn"), ("ffn", "bilstm"), ("cnn", "bilstm")]
t8 = []
for a, b in pairs:
    r = U.mcnemar(y["test"], pred42[a], pred42[b])
    t8.append({"Comparison (A vs B)": f"{NAMES[a]} vs {NAMES[b]}", "A right, B wrong": r["b"], "A wrong, B right": r["c"],
               "Disagree": f"{100 * r['disagree']:.1f}%", "Chi-square": f"{r['chi2']:.2f}", "p": U.fmt_p(r["p"])})
table8 = pd.DataFrame(t8)
U.save_table(table8, "table8_rq1_mcnemar")
table8

### RQ1 decision rule
Recommend the **least costly** model whose mean accuracy is within **one percentage point** of the best model.

In [ ]:
summary = (final.groupby("model").agg(acc=("test_accuracy", "mean"), sd=("test_accuracy", "std"), f1=("test_macro_f1", "mean"))
           .join(lat.set_index("model")))
best_acc = summary.acc.max()
eligible = summary[summary.acc >= best_acc - 0.01].sort_values("cpu_ms_per_review")
RQ1_WINNER = eligible.index[0]
RQ1_BEST_NEURAL = summary.drop("logreg").acc.idxmax()
json.dump(dict(rq1_recommended=RQ1_WINNER, rq1_most_accurate=summary.acc.idxmax(), rq1_best_neural=RQ1_BEST_NEURAL,
               best_accuracy=best_acc), open(f"{RES}/rq1_decision.json", "w"), indent=2)
print(f"Most accurate: {NAMES[summary.acc.idxmax()]} ({100 * best_acc:.2f}%)")
print("Within 1 pp of best:", [NAMES[k] for k in eligible.index])
print(f"RECOMMENDED (least costly within 1 pp): {NAMES[RQ1_WINNER]} — "
      f"{100 * summary.loc[RQ1_WINNER, 'acc']:.2f}% at {summary.loc[RQ1_WINNER, 'cpu_ms_per_review']:.2f} ms/review on CPU")
for k in summary.index:
    ratio = summary.loc[k, "cpu_ms_per_review"] / summary.loc["logreg", "cpu_ms_per_review"]
    print(f"  {NAMES[k]:16s} acc {100 * summary.loc[k, 'acc']:.2f}%  latency x{ratio:.1f} of baseline")

### Figure 6 — Accuracy versus latency (bubble area ∝ parameters)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for k in ["logreg", "ffn", "cnn", "bilstm"]:
    r = summary.loc[k]; params = final[final.model == k].n_params.iloc[0]
    ax.scatter(r.cpu_ms_per_review, 100 * r.acc, s=max(40, params / 1e6 * 60), alpha=0.6, color=U.MODEL_COLORS[NAMES[k]], edgecolor="k", lw=0.6)
    ax.errorbar(r.cpu_ms_per_review, 100 * r.acc, yerr=100 * (0 if pd.isna(r.sd) else r.sd), color="k", lw=0.8, capsize=3)
    ax.annotate(f"{NAMES[k]}\n{params / 1e6:.2f}M params", (r.cpu_ms_per_review, 100 * r.acc), xytext=(12, -4), textcoords="offset points", fontsize=8)
ax.axhspan(100 * (best_acc - 0.01), 100 * best_acc, color="0.85", alpha=0.5, label="Within 1 pp of best")
ax.set_xscale("log"); ax.set_xlabel("End-to-end CPU latency (ms per review, log scale)"); ax.set_ylabel("Test accuracy (%)")
ax.set_title("Figure 6. Accuracy versus latency on the official IMDb test set"); ax.legend(loc="lower right")
U.save_fig(fig, "fig6_accuracy_vs_latency"); plt.show()

### Figure 7 — Training and validation curves (seed 42)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, k in zip(axes, ["ffn", "cnn", "bilstm"]):
    h = pd.read_csv(f"{RES}/rq1_probs/{k}_raw_s42_history.csv")
    ax.plot(h.epoch, h.train_loss, "o-", label="Train loss", color="0.4")
    ax.plot(h.epoch, h.val_loss, "o-", label="Validation loss", color=U.NEG_COLOR)
    ax2 = ax.twinx(); ax2.plot(h.epoch, 100 * h.val_acc, "s--", color=U.POS_COLOR, label="Validation accuracy"); ax2.set_ylabel("Validation accuracy (%)")
    ax2.spines["top"].set_visible(False); ax2.grid(False)
    best_ep = h.val_loss.idxmin(); ax.axvline(h.epoch[best_ep], ls=":", color="0.5")
    ax.set_title(NAMES[k]); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    if k == "ffn":
        lines = ax.get_lines()[:2] + ax2.get_lines(); ax.legend(lines, [l.get_label() for l in lines], fontsize=8)
fig.suptitle("Figure 7. Training curves of the neural models (dotted line = epoch restored by early stopping)", fontweight="bold", y=1.03)
fig.tight_layout(); U.save_fig(fig, "fig7_training_curves"); plt.show()

### Figure 8 — Accuracy on short and long reviews

In [ ]:
g = final.groupby("model")[["test_acc_short", "test_acc_long"]].agg(["mean", "std"]).reindex(["logreg", "ffn", "cnn", "bilstm"])
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(4); w = 0.38
ax.bar(x - w / 2, 100 * g[("test_acc_short", "mean")], w, yerr=100 * g[("test_acc_short", "std")].fillna(0), label=f"Short (≤ {LONG_CUT} tokens)", color="#9fb8d1", capsize=3)
ax.bar(x + w / 2, 100 * g[("test_acc_long", "mean")], w, yerr=100 * g[("test_acc_long", "std")].fillna(0), label=f"Long (> {LONG_CUT} tokens)", color="#2a5d8f", capsize=3)
for i, k in enumerate(g.index):
    d = 100 * (g.loc[k, ("test_acc_long", "mean")] - g.loc[k, ("test_acc_short", "mean")])
    ax.text(i, 100 * g.loc[k, ("test_acc_long", "mean")] + 1.0, f"{d:+.1f} pp", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([NAMES[k] for k in g.index]); ax.set_ylabel("Test accuracy (%)")
lo_ = 100 * min(g[("test_acc_short", "mean")].min(), g[("test_acc_long", "mean")].min())
ax.set_ylim(max(0, lo_ - 8), 100); ax.legend(loc="lower left")
ax.set_title("Figure 8. Accuracy on short and long reviews (mean ± SD across seeds)")
U.save_fig(fig, "fig8_long_vs_short"); plt.show()
print("Long minus short accuracy (pp):", {NAMES[k]: round(100 * (g.loc[k, ("test_acc_long", "mean")] - g.loc[k, ("test_acc_short", "mean")]), 1) for k in g.index})

### Table 9 — Score-masking sensitivity (seed 42)

In [ ]:
sens = []
for k in ["logreg", "ffn", "cnn", "bilstm"]:
    r_raw = runs[runs.key == f"{k}_raw_s42"].iloc[0]; r_m = runs[runs.key == f"{k}_masked_s42"].iloc[0]
    sens.append({"Model": NAMES[k],
                 "Accuracy (raw)": f"{100 * r_raw.test_accuracy:.2f}%", "Accuracy (masked)": f"{100 * r_m.test_accuracy:.2f}%",
                 "Change (pp)": f"{100 * (r_m.test_accuracy - r_raw.test_accuracy):+.2f}",
                 "Score reviews raw": f"{100 * r_raw.test_acc_score_reviews:.1f}%", "Score reviews masked": f"{100 * r_m.test_acc_score_reviews:.1f}%",
                 "Change on score reviews (pp)": f"{100 * (r_m.test_acc_score_reviews - r_raw.test_acc_score_reviews):+.1f}"})
table9 = pd.DataFrame(sens)
U.save_table(table9, "table9_score_masking")
print(f"Test reviews that state a score: {int(meta['test'].has_score.sum()):,} ({meta['test'].has_score.mean():.1%})")
table9

## Summary for the report
* **Table 7 / Figure 6:** accuracy, cost and latency of the four architectures on the official test set, with 5-seed mean ± SD.
* **Table 8:** paired McNemar tests.
* **Decision rule:** the least costly model within one point of the best (printed above, also saved in `rq1_decision.json`).
* **Figure 8:** the long-review hypothesis.
* **Table 9:** sensitivity to explicit score mentions.

**Next:** `04_RQ2_Transfer_Learning.ipynb`.